# JV Curve Analysis Using NOMAD Data

This notebook pulls real JV (current-voltage) measurement data out of NOMAD and turns it into plots -- the same task the **JV Analysis** app automates, but done by hand so you can see every step.

> New to `pandas` (DataFrames, `.groupby`, boolean masks)? `03_pandas_intro.ipynb` covers everything Part 2 assumes -- worth a detour first if that syntax is unfamiliar.

## What You'll Learn

This notebook covers:

1. **Connecting to NOMAD** -- the shared bootstrap and `get_token` login every NOMAD-talking notebook uses
2. **`api_calls.py`** -- the batch → sample IDs → measurement data pattern used by every app
3. **Building a DataFrame from a nested API response** -- the same flattening technique from `03_pandas_intro.ipynb`
4. **Plotly boxplots with `px.box`** -- the exact technique `JV-Analysis`'s own `plot_manager.py` uses
5. **Representative curve selection** -- picking a median-PCE device per condition and plotting it

It has two parts:

- **Part 1 -- Setup**: get this notebook talking to NOMAD (imports + login).
- **Part 2 -- Learning `api_calls.py`**: use this repo's shared query functions to fetch data, then turn it into a `pandas` DataFrame and a couple of Plotly plots.


## 0. Running code cells

- Run a cell with **Shift + Enter**
- Run the cells **in order** -- later cells (e.g. plotting) depend on variables created earlier (e.g. `df_exp`).


---
# Part 1 -- Setup

Everything in this part just gets the notebook ready: no NOMAD-specific knowledge yet, just plumbing.


## 1.1 Connect this notebook to `hysprint_utils`

Every app in this repo shares its NOMAD helper code through one package: `shared/hysprint_utils/`. The cell below installs it into this notebook's kernel so `import hysprint_utils...` works, the same bootstrap every app's own `<app>.ipynb` uses.


In [ ]:
import subprocess
import sys
from pathlib import Path

# Learning/ sits next to shared/ at the repo root
_shared = (Path.cwd() / "../shared").resolve()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(_shared)], check=True)
if str(_shared) not in sys.path:
    sys.path.insert(0, str(_shared))
del _shared

print("hysprint_utils is ready to import.")


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from hysprint_utils.access_token import get_token
from hysprint_utils.api_calls import get_all_JV, get_ids_in_batch, get_sample_description
from hysprint_utils.config import API_ENDPOINT, URL_BASE

print("All imports OK.")


## 1.2 Log in to NOMAD

`get_token(url)` (from `hysprint_utils.access_token`) tries, in order:

1. the `NOMAD_CLIENT_ACCESS_TOKEN` environment variable (set automatically when this notebook runs inside a NOMAD Oasis North tool),
2. a local `secrets.py` with a `NOMAD_TOKEN` variable (see the repo root for the template -- handy for running this notebook on your own machine),
3. and only if neither exists, prompts you for a username and password.


In [ ]:
url = f"{URL_BASE}{API_ENDPOINT}"
token = get_token(url)

print("Token received" if token else "No token -- login failed")


---
# Part 2 -- Learning `api_calls.py`

[`shared/hysprint_utils/api_calls.py`](../shared/hysprint_utils/api_calls.py) is where every app in this repo gets its NOMAD data from -- it's plain functions that take `(url, token, ...)` and return plain Python (lists, dicts, datafram-able records), no widgets involved. Skim the file; the functions used below are only a handful of the ones defined there.

The pattern used across every app is the same three-step funnel:

1. **batch name(s) → sample IDs** -- `get_ids_in_batch(url, token, batch_ids)`
2. **sample IDs → measurement data** -- `get_all_JV(url, token, sample_ids)` (there's a `get_all_eqe`, `get_all_mppt`, ... sibling for every measurement type)
3. **sample IDs → human-readable descriptions** -- `get_sample_description(url, token, sample_ids)`


## 2.1 From a batch name to JV data

`get_ids_in_batch` takes a **list** of batch names, even if you only want one -- that's why `batch_list` below is wrapped in `[...]`. This exact three-call sequence is what [`apps/JV-Analysis/data_manager.py`](../apps/JV-Analysis/data_manager.py) does for every batch you select in the JV Analysis app.


In [ ]:
batch_list = ["HZB_Echem_162_1"]  # ✅ put your own batch name(s) here

sample_ids = get_ids_in_batch(url, token, batch_list)
print(f"Found {len(sample_ids)} samples in {batch_list}")

jv_data = get_all_JV(url, token, sample_ids)
missing = [s for s in sample_ids if s not in jv_data]
print(f"JV data: {len(jv_data)}/{len(sample_ids)} samples found")
if missing:
    print(f"  ⚠️ No JV measurement in NOMAD for: {missing}")

descriptions = get_sample_description(url, token, sample_ids)


## 2.2 Peek at the raw structure before parsing it

`get_all_JV` returns a dict: `{lab_id: [(data, metadata), ...]}`, one `(data, metadata)` pair per JV entry linked to that sample. Before writing parsing code, it's worth printing one entry to see what fields are actually there -- this is a good habit for *any* new NOMAD query, not just this one.


In [ ]:
first_lab_id = next(iter(jv_data))
first_data, first_metadata = jv_data[first_lab_id][0]

print("Top-level fields:", list(first_data.keys()))
print("\nFields on one jv_curve entry:")
print(list(first_data["jv_curve"][0].keys()))


## 2.3 Turn the API response into a tidy DataFrame

Each sample can have several `jv_curve` entries (one per cell, per scan direction). We flatten all of that into one **row per curve**, using the same column names the rest of this repo's apps use (`Voc(V)`, `Jsc(mA/cm2)`, ...) so the DataFrame below plugs straight into familiar plotting code.


In [ ]:
rows = []

for lab_id, entries in jv_data.items():
    condition = descriptions.get(lab_id)

    for data, _metadata in entries:
        for curve in data.get("jv_curve", []):
            cell_name = curve.get("cell_name", "")

            if "_for" in cell_name:
                direction = "Forward"
            elif "_rev" in cell_name:
                direction = "Reverse"
            else:
                continue  # skip curves that are neither forward nor reverse

            J = np.array(curve["current_density"])
            V = np.array(curve["voltage"])

            mask = J <= 28  # adjust threshold as needed
            V, J = V[mask], J[mask]

            rows.append({
                "lab_id": lab_id,
                "condition": condition,
                "direction": direction,
                "cell": cell_name[0],
                "PCE(%)": curve.get("efficiency"),
                "Voc(V)": curve.get("open_circuit_voltage"),
                "Jsc(mA/cm2)": curve.get("short_circuit_current_density"),
                "FF(%)": curve.get("fill_factor"),
                "N_Rs": (curve.get("series_resistance") or 0) * 1e3,
                "N_Rsh": (curve.get("shunt_resistance") or 0) * 1e3,
                "Voltage(V)": V,
                "Current(mA/cm2)": J,
            })

df_exp = pd.DataFrame(rows)
df_exp = df_exp[df_exp["PCE(%)"] > 5]  # drop obviously-failed cells
df_exp.head()


---
# Part 3 -- Plotting the results

Same Plotly idioms as `02_Plotly_plots_intro.ipynb` -- if any of `go.Figure`, `make_subplots`, or `fig.update_layout` looks unfamiliar, that notebook covers them from scratch.


## 3.1 Boxplots of Voc / Jsc / FF / PCE by condition

`plotly.express`'s `px.box(..., points="all")` draws a boxplot with every data point overlaid and jittered -- this is exactly the technique [`apps/JV-Analysis/plot_manager.py`](../apps/JV-Analysis/plot_manager.py) uses for its own box plots. To combine four metrics into one 2×2 figure, we build one `px.box` per metric and copy its traces into a `make_subplots` grid.


In [ ]:
metrics = ["Voc(V)", "Jsc(mA/cm2)", "FF(%)", "PCE(%)"]
positions = [(1, 1), (1, 2), (2, 1), (2, 2)]

fig = make_subplots(rows=2, cols=2, subplot_titles=metrics)

for (row, col), metric in zip(positions, metrics):
    box_fig = px.box(
        df_exp,
        x="condition",
        y=metric,
        color="direction",
        points="all",
        color_discrete_map={"Reverse": "#004f84", "Forward": "#d15e57"},
    )
    for trace in box_fig.data:
        trace.marker.update(size=4, opacity=0.6)
        trace.showlegend = (row, col) == (1, 1)  # only show the legend once
        fig.add_trace(trace, row=row, col=col)

fig.update_layout(height=700, width=900, boxmode="group", title="JV metrics by condition")
fig.update_xaxes(tickangle=-15)
fig.show()


## 3.2 Representative JV curve per condition

For each condition, pick the device closest to the **median PCE**, then plot its forward and reverse curves (solid vs. dotted line).


In [ ]:
fig = go.Figure()
style_map = {"Reverse": "solid", "Forward": "dot"}
palette = px.colors.qualitative.Plotly

for i, (condition, group) in enumerate(df_exp.groupby("condition")):
    med = group["PCE(%)"].median()
    idx = (group["PCE(%)"] - med).abs().idxmin()
    lab_id, cell = group.loc[idx, "lab_id"], group.loc[idx, "cell"]

    device = group[(group["lab_id"] == lab_id) & (group["cell"] == cell)]
    color = palette[i % len(palette)]

    for direction, g2 in device.groupby("direction"):
        V = g2.iloc[0]["Voltage(V)"]
        J = g2.iloc[0]["Current(mA/cm2)"]
        fig.add_trace(go.Scatter(
            x=V, y=J, mode="lines",
            line=dict(color=color, dash=style_map.get(direction, "solid")),
            name=f"{condition} | {direction}",
        ))

fig.add_hline(y=0, line_color="grey", line_width=0.5)
fig.add_vline(x=0, line_color="grey", line_width=0.5)
fig.update_layout(
    title="Representative JV curve per condition (median-PCE device)",
    xaxis_title="Voltage (V)", yaxis_title="Current density (mA/cm²)",
    width=500, height=500,
)
fig.show()

fig.write_html("JV_curves_by_condition.html")
print("Saved: JV_curves_by_condition.html")


## Exercises (recommended)

1. Point `batch_list` at one of your own batches and re-run everything.
2. Add a **5th panel** to the boxplot grid for `N_Rs` (series resistance).
3. `get_all_JV` has siblings -- open `api_calls.py` and try `get_all_eqe` or `get_all_mppt` on the same `sample_ids` list.
4. Change `points="all"` to `points="outliers"` in the boxplot cell and compare.
